# KUMUTEVA fairness analysis

Every figure in the paper's fairness evaluation.

Set `RUN_SELECTION` below to `"latest"` for the most recent run of each
solution, or `"all"` to use every repetition. Everything else stays the same.

The code lives in `utils/` — see the docstrings there for how it works.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd

import utils.analysis as analysis
import utils.display as display
import utils.plots as plots
import utils.stats as stats
import utils.style as style

warnings.filterwarnings("ignore")
style.apply_paper_style()

RAW_DATA_DIR = "./raw"
OUT_DIR = "./"
SYSTEMS = ["control_plane", "network", "storage", "workload"]

# "latest" = most recent run of each solution. "all" = every repetition.
RUN_SELECTION = "all"


## Load and measure

Reads one run at a time so memory stays low, and prints how many runs it found.

Columns to read:

- **delta_latency** — how many times slower the regular tenant got. 1.0 means
  it was not affected.
- **throughput_retention** — how much of its normal work rate it kept. 1.0 means
  all of it.
- **owner_throttled** — the regular tenant could not keep up. When this is true,
  `delta_latency` understates the damage.
- **delta_spread** — the gap between the best and worst repetition. A large gap
  means the runs disagree.
- **failed_runs** — runs that produced no usable data and were left out. Any
  number above zero is worth investigating before trusting the row.

In [ ]:
measurements = []
loaded = {}

for system in SYSTEMS:
    per_system, experiments, report = analysis.load(
        RAW_DATA_DIR, [system], selection=RUN_SELECTION
    )
    if not experiments:
        print(f"{system}: no data")
        continue
    measurements.extend(per_system)
    loaded[system] = experiments
    print(f"{system}: {report.describe()} — {', '.join(plots.ordered_solutions(experiments))}")

# One bar per solution, even when several runs were loaded.
plot_measurements = analysis.measurements_for_plots(measurements, RUN_SELECTION)

summary = analysis.summarise(measurements, RUN_SELECTION)
pd.set_option("display.width", 250)
columns = [c for c in [
    "solution", "system", "runs", "failed_runs",
    "delta_latency", "delta_ci_low", "delta_ci_high",
    "delta_spread", "throughput_retention", "owner_throttled", "stress_err_pct",
] if c in summary.columns]
summary.sort_values(["system", "solution"])[columns]

### Table I values

The `delta_latency` column above, laid out as it appears in the paper.

In [ ]:
table = summary.pivot(index="solution", columns="system", values="delta_latency")
table = table.reindex(sorted(summary.solution.unique(), key=plots.DEFAULT_SOLUTION_ORDER.index))
table.round(2)

## Figures

One tab per subsystem. The figures have no titles because the paper's captions
supply them, so the tab label is the title.

Right-click a figure to save it. To zoom, call the `plots.*` function directly
in a cell of your own.

### Baseline to stress transition

Quiet period first, then the noisy neighbour starts at the marked line. Shows
the moment each solution reacts.

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_latency_transition(loaded[system], system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

### Latency over time

The regular tenant while the noisy neighbour is running.

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_latency_over_time(loaded[system], system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

### Latency distribution by phase

Three bars per solution: the regular tenant when quiet, the regular tenant under
stress, and the noisy neighbour. Only the neighbour's load is increased.

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_latency_distribution(loaded[system], system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

### 95th percentile latency

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_p95_comparison(loaded[system], system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

### Latency density

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_latency_violin(loaded[system], system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

### Degradation factor

How many times slower the regular tenant got. Hatched bars are cases where it
also could not keep up, so the real damage is worse than the bar shows.

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_degradation(plot_measurements, system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

### Throughput retention

How much of its normal work rate the regular tenant kept. Read together with the
degradation factor: a tenant can be hurt by getting slower, by getting less
done, or both.

In [ ]:
display.figure_tabs(
    lambda system: plots.plot_throughput_retention(plot_measurements, system, save=False),
    [s for s in SYSTEMS if s in loaded],
    labels=plots.SYSTEM_NAMES,
)

## Export figures for the paper

Nothing above writes files. Run this when the figures are final — it is slow,
because PGF output runs LaTeX.

In [ ]:
EXPORT_FORMATS = ("png", "pgf")  # drop "pgf" to skip LaTeX entirely

exporters = {
    "latency_transition": lambda s: plots.plot_latency_transition(loaded[s], s, save=False),
    "latency_over_time": lambda s: plots.plot_latency_over_time(loaded[s], s, save=False),
    "latency_distribution": lambda s: plots.plot_latency_distribution(loaded[s], s, save=False),
    "p95_comparison": lambda s: plots.plot_p95_comparison(loaded[s], s, save=False),
    "latency_violin": lambda s: plots.plot_latency_violin(loaded[s], s, save=False),
    "degradation": lambda s: plots.plot_degradation(plot_measurements, s, save=False),
    "throughput_retention": lambda s: plots.plot_throughput_retention(plot_measurements, s, save=False),
}

for system in loaded:
    for name, build in exporters.items():
        figure, _ = build(system)
        style.save_plot(figure, f"{OUT_DIR}/{system}_{name}", formats=EXPORT_FORMATS)
        plt.close(figure)
    print("exported", system)

## Per-run detail

Every individual run, so you can spot one that went wrong. The table at the top
averages these.

In [ ]:
per_run = stats.to_frame(measurements).sort_values(["system", "solution", "delta_latency"])
per_run[["solution", "system", "delta_latency", "throughput_retention", "baseline_ms", "stress_ms"]].round(4)